# **Vietnamese Text Summarization - Progressive Training Pipeline**

Quy trình huấn luyện nâng cao:
- **Phase 1**: Full fine-tune trên dữ liệu tổng quát
- **Phase 2a**: Freeze encoder, chỉ train decoder trên dữ liệu chuyên ngành (Medical)
- **Phase 2b**: Unfreeze toàn bộ, train với LR rất thấp trên dữ liệu chuyên ngành

### Hướng dẫn sử dụng:
1. Bật **GPU T4 x2** (hoặc P100) trên Kaggle
2. Chạy lần lượt các cell từ trên xuống dưới.
3. Thời gian dự kiến: ~3.5h - 4h.

In [ ]:
from __future__ import annotations
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

In [ ]:
REPO_URL = "https://github.com/dungcony/sumarization.git"

if os.path.exists(".git") and "sumarization" in os.getcwd():
    print("Đang cập nhật code...")
    !git pull
else:
    print("Đang tải mã nguồn...")
    !git clone {REPO_URL} sumarization
    %cd sumarization

In [ ]:
%pip install -e . 2>&1 | tail -5

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import threading
import csv
from pathlib import Path
import torch

from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
)

from src.config import load_config, apply_overrides, config_to_dict
from src.data import load_and_preprocess
from src.evaluator import build_compute_metrics
from src.model import (
    apply_lora,
    enable_gradient_checkpointing,
    freeze_encoder,
    load_model,
    load_tokenizer,
)
from src.utils import (
    format_duration,
    format_number,
    save_json,
    set_seed,
    setup_logger,
    count_parameters,
)

logger = setup_logger("notebook")
print("✅ Import thành công!")

## Callback Custom cho Logging

In [ ]:
class KaggleProgressCallback(TrainerCallback):
    def __init__(self, label, log_every_steps=10, heartbeat_seconds=60, log_file=None, raw_dataset=None, tokenizer=None):
        self.label = label
        self.log_every_steps = max(1, int(log_every_steps))
        self.heartbeat_seconds = max(10, int(heartbeat_seconds))
        self.log_file = Path(log_file) if log_file else None
        self.raw_dataset = raw_dataset
        self.tokenizer = tokenizer
        self.started_at = None
        self.last_completed_step = 0
        self.max_steps = 0
        self._is_main_process = True
        self._stop_event = threading.Event()
        self._write_lock = threading.Lock()
        self._heartbeat_thread = None
        self._last_eval_loss = None

    def _device_status(self):
        if torch.cuda.is_available():
            allocated = torch.cuda.memory_allocated() / (1024 ** 3)
            reserved = torch.cuda.memory_reserved() / (1024 ** 3)
            return f"GPU RAM={allocated:.2f}/{reserved:.2f} GB"
        return "CPU"

    def _emit(self, message):
        if not self._is_main_process: return
        timestamp = time.strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] [{self.label}] {message}"
        with self._write_lock:
            print(line, flush=True)
            if self.log_file:
                self.log_file.parent.mkdir(parents=True, exist_ok=True)
                with self.log_file.open("a", encoding="utf-8") as handle:
                    handle.write(line + "\n")

    def _heartbeat_loop(self):
        while not self._stop_event.wait(self.heartbeat_seconds):
            self._emit(f"♥ VẪN ĐANG TRAIN | step={self.last_completed_step}/{self.max_steps} | {self._device_status()}")

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.time()
        self.max_steps = state.max_steps
        self._is_main_process = state.is_world_process_zero
        self._stop_event.clear()
        if self._is_main_process:
            self._heartbeat_thread = threading.Thread(target=self._heartbeat_loop, daemon=True)
            self._heartbeat_thread.start()

    def on_step_end(self, args, state, control, **kwargs):
        self.last_completed_step = state.global_step

    def on_log(self, args, state, control, logs=None, **kwargs):
        logs = logs or {}
        fields = [f"{k}={v:.6g}" if isinstance(v, float) else f"{k}={v}" for k, v in logs.items() if k in ("loss", "learning_rate", "epoch")]
        if fields: self._emit("METRICS | " + " | ".join(fields))
        
        train_loss = logs.get("loss")
        if self._last_eval_loss and train_loss and train_loss > 0:
            ratio = self._last_eval_loss / train_loss
            if ratio > 1.5:
                self._emit(f"⚠️ CẢNH BÁO OVERFITTING: eval_loss/train_loss = {ratio:.2f}")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        self._last_eval_loss = metrics.get("eval_loss")
        
        if self.log_file:
            csv_path = self.log_file.parent / "eval_history.csv"
            row = { "step": state.global_step, "epoch": round(state.epoch, 2) if state.epoch else 0, **{k:v for k,v in metrics.items() if k.startswith("eval_")} }
            write_header = not csv_path.exists()
            with open(csv_path, "a", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=row.keys())
                if write_header: writer.writeheader()
                writer.writerow(row)

        summary = ", ".join(f"{k}={v:.4f}" for k, v in metrics.items() if k in ("eval_loss", "eval_rougeL", "eval_gen_len"))
        self._emit(f"ĐÁNH GIÁ XONG | {summary}")

        # In mẫu sinh text
        if self.raw_dataset and self.tokenizer and kwargs.get('model'):
            self._print_samples(kwargs['model'])

    def _print_samples(self, model, n=2):
        model.eval()
        device = next(model.parameters()).device
        for i in range(min(n, len(self.raw_dataset))):
            article = self.raw_dataset[i]["article"][:400]
            ref = self.raw_dataset[i]["summary"]
            inputs = self.tokenizer("summarize: " + article, max_length=768, truncation=True, return_tensors="pt").to(device)
            with torch.no_grad():
                out = model.generate(**inputs, max_length=128, num_beams=2)
            pred = self.tokenizer.decode(out[0], skip_special_tokens=True)
            self._emit(f"\n--- Mẫu {i+1} ---\n📄 Gốc: {article[:150]}...\n✅ Ref: {ref}\n🤖 Gen: {pred}")

    def stop(self):
        self._stop_event.set()
        if self._heartbeat_thread and self._heartbeat_thread.is_alive():
            self._heartbeat_thread.join(timeout=2)
            
    def on_train_end(self, args, state, control, **kwargs):
        self._emit("KẾT THÚC TRAIN")
        self.stop()


## Hàm Helper Train Phase

In [ ]:
from datasets import load_dataset
from src.data import load_dataset_from_files

def run_phase(config_file, model_override=None, output_override=None):
    print(f"\n{'='*80}\n🚀 BẮT ĐẦU: {config_file}\n{'='*80}")
    config = load_config(config_file)
    
    overrides = {}
    if model_override: overrides["model.name_or_path"] = model_override
    if output_override: overrides["training.output_dir"] = output_override
    
    # Kaggle T4 optimization
    overrides["training.precision"] = "fp16"
    overrides["training.optim"] = "adafactor"
    overrides["training.label_smoothing_factor"] = 0.0
    overrides["generation.num_beams"] = 2
    
    config = apply_overrides(config, overrides)
    output_dir = Path("/kaggle/working") / config.training.output_dir if Path("/kaggle").exists() else Path(config.training.output_dir)
    best_dir = output_dir / "best"
    
    set_seed(config.training.seed)
    tokenizer = load_tokenizer(config.model)
    model = load_model(config.model, tokenizer, config.generation)
    
    if config.training.freeze_encoder:
        freeze_encoder(model)
    
    datasets = load_and_preprocess(tokenizer, config.data)
    raw_valid_dataset = load_dataset_from_files(config.data.train_file, config.data.valid_file)["validation"]
    
    tc = config.training
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(output_dir),
        seed=tc.seed,
        num_train_epochs=tc.num_train_epochs,
        per_device_train_batch_size=tc.per_device_train_batch_size,
        per_device_eval_batch_size=2, # Giữ thấp để không OOM khi sinh text
        gradient_accumulation_steps=tc.gradient_accumulation_steps,
        learning_rate=tc.learning_rate,
        weight_decay=tc.weight_decay,
        warmup_ratio=tc.warmup_ratio,
        lr_scheduler_type=tc.lr_scheduler_type,
        optim=tc.optim,
        label_smoothing_factor=tc.label_smoothing_factor,
        fp16=(tc.precision == "fp16"),
        eval_strategy=tc.eval_strategy,
        eval_steps=tc.eval_steps,
        save_strategy=tc.save_strategy,
        save_steps=tc.save_steps,
        save_total_limit=tc.save_total_limit,
        logging_steps=tc.logging_steps,
        metric_for_best_model=tc.metric_for_best_model,
        greater_is_better=tc.greater_is_better,
        load_best_model_at_end=tc.load_best_model_at_end,
        predict_with_generate=True,
        generation_max_length=config.generation.max_length,
        report_to=["tensorboard"],
        disable_tqdm=True,
    )
    
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100)
    
    callbacks = [KaggleProgressCallback(
        label=config.phase.name, 
        log_every_steps=tc.logging_steps, 
        log_file=output_dir / "training.log",
        raw_dataset=raw_valid_dataset,
        tokenizer=tokenizer
    )]
    if tc.early_stopping_patience > 0:
        callbacks.append(EarlyStoppingCallback(early_stopping_patience=tc.early_stopping_patience))
        
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=datasets["train"],
        eval_dataset=datasets["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=build_compute_metrics(tokenizer),
        callbacks=callbacks,
    )
    
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    
    try:
        trainer.train()
    finally:
        callbacks[0].stop()
        
    trainer.save_model(str(best_dir))
    tokenizer.save_pretrained(str(best_dir))
    
    eval_res = trainer.evaluate(metric_key_prefix="eval")
    save_json(eval_res, output_dir / "eval_results.json")
    
    import gc; del trainer; del model; gc.collect(); torch.cuda.empty_cache()
    return str(best_dir), eval_res


## 🏃‍♂️ Chạy Phase 1: Full Fine-tune (General Data)

In [ ]:
best_phase1, eval_p1 = run_phase("configs/vit5_base_phase_1.yaml")
print(f"Phase 1 hoàn thành. Checkpoint: {best_phase1}")
print(f"ROUGE-L Phase 1: {eval_p1['eval_rougeL']}")

## 🏃‍♂️ Chạy Phase 2a: Freeze Encoder (Medical Data)

In [ ]:
best_phase2a, eval_p2a = run_phase(
    "configs/vit5_base_phase_2a.yaml", 
    model_override=best_phase1
)
print(f"Phase 2a hoàn thành. Checkpoint: {best_phase2a}")
print(f"ROUGE-L Phase 2a: {eval_p2a['eval_rougeL']}")

## 🏃‍♂️ Chạy Phase 2b: Unfreeze All, Low LR (Medical Data)

In [ ]:
best_phase2b, eval_p2b = run_phase(
    "configs/vit5_base_phase_2b.yaml", 
    model_override=best_phase2a
)
print(f"Phase 2b hoàn thành. Checkpoint: {best_phase2b}")
print(f"ROUGE-L Phase 2b: {eval_p2b['eval_rougeL']}")

## 🎯 Đánh Giá Chéo (Cross-phase Evaluation)

In [ ]:
from src.evaluator import evaluate_checkpoint

print("Đang đánh giá Phase 2b model trên data Phase 1 (kiểm tra Forgetting)...")

config_phase1 = load_config("configs/vit5_base_phase_1.yaml")

cross_eval_dir = Path("/kaggle/working/outputs/vit5_base_phase_2b/cross_eval") if Path("/kaggle").exists() else Path("outputs/vit5_base_phase_2b/cross_eval")
cross_eval = evaluate_checkpoint(
    model_path=best_phase2b, 
    config=config_phase1,
    output_dir=str(cross_eval_dir)
)

print("\n=== KẾT QUẢ FORGETTING CHECK ===")
print(f"Phase 1 Model trên Phase 1 Data: ROUGE-L = {eval_p1['eval_rougeL']:.2f}")
print(f"Phase 2b Model trên Phase 1 Data: ROUGE-L = {cross_eval['eval_rougeL']:.2f}")
print(f"Chênh lệch: {cross_eval['eval_rougeL'] - eval_p1['eval_rougeL']:.2f}")
if (cross_eval['eval_rougeL'] - eval_p1['eval_rougeL']) > -5.0:
    print("✅ Không bị catastrophic forgetting nghiêm trọng!")
else:
    print("⚠️ Cảnh báo: Model quên khá nhiều kiến thức Phase 1.")
